In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/lopezjatjat10@gmail.com/sportsbar-etl-proj/1_setup/utilities

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'products','Data Source')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')


In [0]:

df_bronze = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source}')

display(df_bronze)

In [0]:
df_bronze.printSchema()

In [0]:
df_duplicated = df_bronze.groupBy('product_id').count().filter(F.col('count') > 1)

display(df_duplicated)

In [0]:
print(f'Row Count Before Dropping Duplicates : {df_bronze.count()}')
df_silver = df_bronze.dropDuplicates(['product_id'])
print(f'Row Count After Dropping Duplicates : {df_silver.count()}')

In [0]:
df_silver.select('category').distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    'category', 
    F.when(F.col('category').isNull(), None)
    .otherwise(F.initcap('category'))
)

In [0]:
df_silver.select('category').distinct().show()

In [0]:
df_silver = df_silver.withColumn(
            'product_name',
            F.regexp_replace(F.col('product_name'), '(?i)protien', 'Protein')
        ).withColumn(
            'category',
            F.regexp_replace(F.col('category'), '(?i)protien', 'Protein')
        )

In [0]:
display(df_silver.limit(10))

In [0]:
df_silver = df_silver.withColumn(
    'division',
    F.when(F.col("category") == "Energy Bars",        "Nutrition Bars")
         .when(F.col("category") == "Protein Bars",       "Nutrition Bars")
         .when(F.col("category") == "Granola & Cereals",  "Breakfast Foods")
         .when(F.col("category") == "Recovery Dairy",     "Dairy & Recovery")
         .when(F.col("category") == "Healthy Snacks",     "Healthy Snacks")
         .when(F.col("category") == "Electrolyte Mix",    "Hydration & Electrolytes")
         .otherwise("Other")
)

In [0]:
df_silver = df_silver.withColumn(
    'variant',
    F.regexp_extract(F.col('product_name'), r'\((.*?)\)', 1)
)

In [0]:
df_silver = df_silver.withColumn(
    'product_code',
    F.sha2(F.col("product_name").cast('string'), 256)
).withColumn(
    'product_id',
    F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit(999999).cast("string"))
).withColumnRenamed('product_name', 'product')




In [0]:
display(df_silver)

In [0]:
df_silver.printSchema()

In [0]:
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "file_name", "file_size")

display(df_silver)

In [0]:
df_silver.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', 'true')\
    .option('mergeSchema', 'true')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
